# 🚦 TRAFİK İŞARETİ SINIFLANDIRMA SİSTEMİ (CNN)

**Geliştiren:** Batuhan

**MİMARİ ÖZELLİKLERİ:**
1. Çok Katmanlı CNN Yapısı (32, 64, 128 ve 256 filtreli Conv2D)
2. Maksimum Havuzlama (MaxPool2D ile boyut indirgeme)
3. ReLU ve Softmax Aktivasyon Fonksiyonları
4. Veri Normalizasyonu (0-1 aralığı)
5. Data Augmentation (Veri çoğaltma ile overfitting önleme)
6. Batch Normalization (Katman bazlı stabilizasyon)
7. Güçlü Dropout Regularizasyonu (%25 ve %50 oranlı)
8. Early Stopping (Erken durdurma)
9. Learning Rate Scheduling (Dinamik öğrenme hızı)
10. Adam Optimizasyonu ve Categorical Crossentropy Loss

# 1. Kütüphane İmportları

In [ ]:
# Temel kütüphaneler
import numpy as np 
import pandas as pd 
import tensorflow as tf
import os

# Görüntü işleme kütüphaneleri
import cv2
from PIL import Image

# Performans ölçme ve görselleştirme
from sklearn import metrics 
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Model ve eğitim kütüphaneleri
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Zaman
import time
import datetime

print('=' * 60)
print('🚦 TRAFİK İŞARETİ SINIFLANDIRMA SİSTEMİ')
print('=' * 60)
print(f'TensorFlow versiyonu: {tf.__version__}')
print(f'Çalışma zamanı: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

# 2. Yardımcı Fonksiyonlar

In [ ]:
# date_time fonksiyonu
def date_time(x):
    if x==1:
        return 'Timestamp: {:%Y-%m-%d %H:%M:%S}'.format(datetime.datetime.now())
    if x==2:    
        return 'Timestamp: {:%Y-%b-%d %H:%M:%S}'.format(datetime.datetime.now())
    if x==3:  
        return 'Date now: %s' % datetime.datetime.now()
    if x==4:  
        return 'Date today: %s' % datetime.date.today()

In [ ]:
# Performance Plot fonksiyonu
def plot_performance(history=None, figure_directory=None, ylim_pad=[0, 0]):
    xlabel = 'Epoch'
    legends = ['Training', 'Validation']

    plt.figure(figsize=(20, 5))

    y1 = history.history['accuracy']
    y2 = history.history['val_accuracy']

    min_y = min(min(y1), min(y2))-ylim_pad[0]
    max_y = max(max(y1), max(y2))+ylim_pad[0]

    plt.subplot(121)
    plt.plot(y1)
    plt.plot(y2)
    plt.title('Model Accuracy\n'+date_time(1), fontsize=17)
    plt.xlabel(xlabel, fontsize=15)
    plt.ylabel('Accuracy', fontsize=15)
    plt.ylim(min_y, max_y)
    plt.legend(legends, loc='upper left')
    plt.grid()

    y1 = history.history['loss']
    y2 = history.history['val_loss']

    min_y = min(min(y1), min(y2))-ylim_pad[1]
    max_y = max(max(y1), max(y2))+ylim_pad[1]

    plt.subplot(122)
    plt.plot(y1)
    plt.plot(y2)
    plt.title('Model Loss\n'+date_time(1), fontsize=17)
    plt.xlabel(xlabel, fontsize=15)
    plt.ylabel('Loss', fontsize=15)
    plt.ylim(min_y, max_y)
    plt.legend(legends, loc='upper left')
    plt.grid()
    if figure_directory:
        plt.savefig(figure_directory+'/history')

    plt.show()

# 3. Veri Yükleme (Loading Dataset)

In [ ]:
data = []
labels = []
classes = 43
cur_path = os.getcwd()

for i in range(classes):
    path = os.path.join('/kaggle/input/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign/Train', str(i))
    images = os.listdir(path)
    for a in images:
        try:
            image = Image.open(path + '/'+ a)
            image = image.resize((30,30))
            image = np.array(image)
            data.append(image)
            labels.append(i)
        except:
            print('Error loading image')

data = np.array(data)
labels = np.array(labels)

print(f'✅ Toplam yüklenen görüntü: {data.shape[0]}')
print(f'✅ Görüntü boyutu: {data.shape[1]}x{data.shape[2]} piksel, {data.shape[3]} kanal (RGB)')
print(f'✅ Sınıf sayısı: {classes}')

# 4. Veri Bölme ve Ön İşleme (Data Splitting)

In [ ]:
# Veri boyutunu kontrol et
print(data.shape, labels.shape)

# Veriyi %80 eğitim, %20 test olarak böl
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

# Normalizasyon
# Piksel değerlerini 0-255 arasından 0-1 arasına indirger.
X_train = X_train / 255.0
X_test = X_test / 255.0

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print('✅ Normalizasyon uygulandı (0-255 → 0-1 aralığına)')

# One-hot encoding
y_train_encoded = to_categorical(y_train, 43)
y_test_encoded = to_categorical(y_test, 43)
print('✅ One-hot encoding uygulandı')

# 5. Data Augmentation (Veri Çoğaltma)

Data Augmentation, eğitim verilerini yapay olarak çoğaltır. Fotoğrafları döndürme, kaydırma, zoom yaparak modelin farklı koşullarda da doğru tahmin yapmasını sağlar. Bu teknik overfitting'i azaltır.

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=15,      # Rastgele ±15 derece döndürme
    width_shift_range=0.1,  # Yatay %10 kaydırma
    height_shift_range=0.1, # Dikey %10 kaydırma
    zoom_range=0.1,         # %10 yakınlaştırma/uzaklaştırma
    shear_range=0.1,        # %10 eğme
    fill_mode='nearest'     # Boş pikselleri en yakın komşu ile doldur
)
datagen.fit(X_train)

print('✅ Data Augmentation aktif: rotation=±15°, shift=%10, zoom=%10')
print('   → Eğitim sırasında her görüntü rastgele dönüştürülecek')

# 6. CNN Modeli Oluşturma

In [ ]:
# Modelin inşası
model = Sequential()

# --- BLOK 1 ---
model.add(Conv2D(filters=32, kernel_size=(5,5), activation='relu', input_shape=X_train.shape[1:]))
model.add(BatchNormalization())
model.add(Conv2D(filters=64, kernel_size=(5,5), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))

# --- BLOK 2 ---
model.add(Conv2D(filters=128, kernel_size=(3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(filters=256, kernel_size=(3, 3), activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))

# --- BLOK 3: Sınıflandırma ---
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(rate=0.5))
model.add(Dense(43, activation='softmax'))

# Modelin derlenmesi
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

# 7. Callbacks Ayarlama

- **Early Stopping:** Validation accuracy 10 epoch iyileşmezse eğitimi durdurur
- **ReduceLROnPlateau:** Validation loss 5 epoch düşmezse learning rate'i yarıya indirir

In [ ]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

print('✅ Early Stopping: patience=10')
print('✅ LR Scheduling: factor=0.5, patience=5')

# 8. Modeli Eğit (Training)

In [ ]:
start_time = time.time()

history = model.fit(
    datagen.flow(X_train, y_train_encoded, batch_size=128),
    epochs=50,
    validation_data=(X_test, y_test_encoded),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

end_time = time.time()
training_time = end_time - start_time

print()
print('=' * 60)
print(f'✅ Eğitim tamamlandı!')
print(f'⏱️  Toplam eğitim süresi: {training_time/60:.1f} dakika')
print(f'📈 En iyi eğitim doğruluğu: {max(history.history["accuracy"])*100:.2f}%')
print(f'📈 En iyi doğrulama doğruluğu: {max(history.history["val_accuracy"])*100:.2f}%')
print('=' * 60)

# 9. Eğitim Grafikleri (Performance Plot)

In [ ]:
plot_performance(history)

# 10. Test Verisi ile Değerlendirme

In [ ]:
# Tahmin yap
pred = np.argmax(model.predict(X_test), axis=-1)

# Accuracy hesapla
test_accuracy = accuracy_score(y_test, pred)
print(f'🎯 TEST DOĞRULUĞU: %{round(test_accuracy * 100, 2)}')

# F1 Skoru
f1 = f1_score(y_test, pred, average='weighted')
print(f'📊 Ağırlıklı F1 Skoru: %{round(f1 * 100, 2)}')

# Detaylı sınıflandırma raporu
print()
print('📋 SINIFLANDIRMA RAPORU:')
print('=' * 60)
print(classification_report(y_test, pred))

# 11. Confusion Matrix (Karışıklık Matrisi)

In [ ]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(18, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(43), yticklabels=range(43))
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Tahmin Edilen Sınıf', fontsize=14)
plt.ylabel('Gerçek Sınıf', fontsize=14)
plt.tight_layout()
plt.show()

# 12. Sonuç Özeti

In [ ]:
# Yüzde değerlerini yuvarla
accuracy_pct = round(test_accuracy * 100, 2)
f1_pct = round(f1 * 100, 2)

print()
print('=' * 60)
print('📊 FİNAL SONUÇLARI:')
print(f'   Model:        CNN Mimarisi')
print(f'   Accuracy:     %{accuracy_pct}')
print(f'   F1-Score:     %{f1_pct}')
print(f'   Eğitim Süresi: {training_time/60:.1f} dakika')
print('=' * 60)
print()
print('  MODEL ÖZELLİKLERİ:')
print('    1. ✅ Çok Katmanlı CNN Yapısı (Conv2D)')
print('    2. ✅ Maksimum Havuzlama (MaxPool2D)')
print('    3. ✅ ReLU ve Softmax Aktivasyon Fonksiyonları')
print('    4. ✅ Veri Normalizasyonu')
print('    5. ✅ Data Augmentation (döndürme, kaydırma, zoom)')
print('    6. ✅ Batch Normalization')
print('    7. ✅ Güçlü Dropout Regularizasyonu')
print('    8. ✅ Early Stopping (overfitting önleme)')
print('    9. ✅ Learning Rate Scheduling')
print('    10. ✅ Adam Optimizer & Categorical Crossentropy')
print()
print('=' * 60)

# 13. Modeli Kaydet

In [ ]:
model.save('traffic_classifier_model.h5')
print('💾 Model kaydedildi: traffic_classifier_model.h5')
print()
print('🎉 Tüm işlemler tamamlandı!')